# Conventional CNN with ROI Extraction

In [ ]:
# Standard library imports
import os
import json
import math

# Data manipulation and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Image processing
from PIL import Image
import cv2
import matplotlib.image as mpimg  # For loading images

# Machine learning libraries
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# TensorFlow and Keras
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

import albumentations as A
from albumentations.pytorch.transforms import ToTensorV2

In [ ]:
def create_dummy_image():
    dummy_image_path = "../dataset/nosample.png" 
    if os.path.exists(dummy_image_path):
        return mpimg.imread(dummy_image_path)
    else:
        print(f"Warning: '{dummy_image_path}' not found. Using a blank dummy image.")
        return np.ones((100, 100, 3), dtype=np.uint8) * 255

In [ ]:
# Define the directory containing the labeled dataset
directory = r'C:\Users\david\Desktop\project\dataset\Labeled'

# Initialize an empty list to store data
data = []

# Iterate through all files in the directory
for file in os.listdir(directory):
    if file.lower().endswith('.jpg'):
        image_path = os.path.join(directory, file)
        json_path = image_path.replace('.jpg', '.json')
        
        label = None
        if os.path.exists(json_path):
            try:
                with open(json_path, 'r') as f:
                    content = json.load(f)
                    shapes = content.get('shapes', [])
                    if shapes and isinstance(shapes, list):
                        label = shapes[0].get('label')
            except Exception as e:
                print(f"Error reading {json_path}: {e}")
        
        data.append({'filename': image_path, 'label': label})

df = pd.DataFrame(data)
print(df.info())
display(df.head())

In [ ]:
def extract_roi(image_path, json_path):
    """
    Extracts the region of interest (ROI) from an image based on bounding box
    coordinates found in the corresponding JSON file.

    Args:
        image_path (str): Path to the image file.
        json_path (str): Path to the JSON file containing bounding box coordinates.

    Returns:
        numpy.ndarray: The ROI extracted from the image. Returns the original
                       image if ROI extraction fails or if the JSON file does
                       not contain valid bounding box information.
    """
    try:
        with open(json_path, 'r') as f:
            content = json.load(f)
            shapes = content.get('shapes', [])
            if shapes and isinstance(shapes, list) and len(shapes) > 0:
                shape = shapes[0]
                if shape.get('shape_type') == 'rectangle' and 'points' in shape:
                    points = shape['points']
                    x1, y1 = map(int, points[0])
                    x2, y2 = map(int, points[1])

                    # Read the image
                    image = cv2.imread(image_path)
                    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

                    # Extract the ROI
                    roi = image[y1:y2, x1:x2]
                    return roi
                else:
                    print(f"Invalid shape type or points in {json_path}")
                    image = cv2.imread(image_path)
                    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                    return image
            else:
                print(f"No valid shapes found in {json_path}")
                image = cv2.imread(image_path)
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                return image
    except Exception as e:
        print(f"Error extracting ROI from {image_path}: {e}")
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        return image

In [ ]:
# Preprocessing
df_filtered = df[~df['label'].isin(['good', 'no_good'])].copy()
target_size = (224, 224)
batch_size = 32

train_df, val_df = train_test_split(
    df_filtered, test_size=0.2, stratify=df_filtered['label'], random_state=42
)

# Define class indices manually
class_indices = {label: idx for idx, label in enumerate(train_df['label'].unique())}

# Define Albumentations augmentation pipeline
augmentation_pipeline = A.Compose([
    A.Resize(target_size[0], target_size[1]),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# Custom data generator using Albumentations
def albumentations_data_generator(df, batch_size, augmentations, target_size):
    while True:
        for i in range(0, len(df), batch_size):
            batch_df = df.iloc[i:i+batch_size]
            images = []
            labels = []
            for _, row in batch_df.iterrows():
                # Read image
                image_path = row['filename']
                json_path = image_path.replace('.jpg', '.json')
                image = extract_roi(image_path, json_path)
                
                # Apply augmentations
                augmented = augmentations(image=image)
                image = augmented['image']
                
                # Transpose image to channels-last format
                if image.shape[0] == 3:  # Check if the image is in channels-first format
                    image = np.transpose(image, (1, 2, 0))  # Convert to (height, width, channels)
                
                images.append(image)
                
                # Map string label to integer and one-hot encode
                label = class_indices[row['label']]  # Convert string label to integer
                labels.append(tf.keras.utils.to_categorical(label, num_classes=len(class_indices)))
            
            yield np.array(images), np.array(labels)

# Create generators
train_generator = albumentations_data_generator(train_df, batch_size, augmentation_pipeline, target_size)
val_generator = albumentations_data_generator(val_df, batch_size, augmentation_pipeline, target_size)

# CNN

In [ ]:
model = models.Sequential([
    layers.Conv2D(16, (3, 3), activation='relu', input_shape=(target_size[0], target_size[1], 3)),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(32, (3, 3), activation='relu'), 
    layers.MaxPooling2D(2, 2),
    layers.Flatten(),
    layers.Dense(64, activation='relu'), 
    # layers.Dense(len(train_generator.class_indices), activation='softmax')
    layers.Dense(len(class_indices), activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Calculate steps per epoch
steps_per_epoch = math.ceil(len(train_df) / batch_size)
validation_steps = math.ceil(len(val_df) / batch_size)

# Train the model
history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,  # Specify steps per epoch
    epochs=10,
    validation_data=val_generator,
    validation_steps=validation_steps  # Specify validation steps
)

# Save the trained model
model.save("../model/conventional_cnn_roi.h5")

In [ ]:
import matplotlib.pyplot as plt

# Plot training & validation accuracy values
plt.figure(figsize=(12, 4))

# Accuracy plot
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')

# Show the plots
plt.tight_layout()
plt.show()

# Evaluation

In [ ]:
# Predict
val_steps = math.ceil(len(val_df) / batch_size)  # Calculate validation steps
predictions = model.predict(val_generator, steps=val_steps)
predicted_classes = np.argmax(predictions, axis=1)

# Ground truth labels
true_classes = val_df['label'].map(class_indices).values  # Map string labels to integers
class_labels = list(class_indices.keys())

# Classification report
report = classification_report(true_classes, predicted_classes, target_names=class_labels)
print("Classification Report:\n", report)

# Confusion matrix
cm = confusion_matrix(true_classes, predicted_classes)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
            xticklabels=class_labels, yticklabels=class_labels)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

# Error analysis: Sample one image from each confusion matrix cell
plt.figure(figsize=(15, 15))
num_classes = len(class_labels)

# Track the number of subplots dynamically
subplot_count = 0

for true_label in range(num_classes):
    for predicted_label in range(num_classes):
        # Find indices of samples for the current true and predicted label pair
        indices = np.where((true_classes == true_label) & (predicted_classes == predicted_label))[0]
        
        if len(indices) > 0:
            # Sample one image from the indices
            idx = indices[0]  # Take the first one (or random.choice(indices) for randomness)
            image_path = val_df['filename'].iloc[idx]

            # Load the image
            if os.path.exists(image_path):
                image = plt.imread(image_path)
            else:
                image = create_dummy_image()
        else:
            # Use a placeholder image if no sample exists for this cell
            image = create_dummy_image()
        
        # Display the image
        subplot_count += 1
        plt.subplot(num_classes, num_classes, subplot_count)
        plt.imshow(image)
        title = f"True: {class_labels[true_label]}\nPred: {class_labels[predicted_label]}"
        if len(indices) == 0:
            title += "\n(No Sample)"
        plt.title(title)
        plt.axis('off')

# Adjust layout and display the plot
if subplot_count > 0:
    plt.tight_layout()
    plt.show()
else:
    print("No misclassifications to display.")